# 실습 01 · AI 서비스 구성요소 연결 (기본필수)

11/10~12에 만든 지식 관리 서비스(FastAPI + PostgreSQL)에 앞으로 LLM이 붙으면 **각 구성요소가 무슨 일을 하고 어떤 순서로 연결되는지**를 코드로 정리하고 그림으로 그립니다. LLM을 실제로 호출하지는 않습니다.

### 사용 방법
1. **결과는 이미 셀 아래에 저장되어 있습니다.** 읽기만 해도 흐름을 따라갈 수 있어요.
2. 직접 돌려 보려면 `파일 → Drive에 사본 저장`으로 내 사본을 만드세요.
3. 맨 위 **⚙️ 준비 셀을 먼저 한 번 실행**한 뒤, 보고 싶은 구간의 셀을 실행하세요. 각 구간은 준비 셀만 있으면 따로 실행됩니다.
4. **✏️ 표시 셀**은 오른쪽 입력 칸의 값을 바꾸고 실행해 보세요. 코드의 값도 함께 바뀝니다.

| 구간 | 내용 |
|---|---|
| B01 | 구성요소 4개의 역할 |
| B02 | 요청 → 응답 연결 순서 (구현됨 / 후속 구분) |
| B03 | Mermaid 그림 만들기 |
| 저장 | 결과 저장 (`main.py`와 같은 결과) |

> 그림 렌더링은 공개 서비스 mermaid.ink 이미지를 사용하므로 인터넷 연결이 필요합니다.

In [1]:
#@title ⚙️ 먼저 실행하세요 — 실습 공통 코드 준비
# ── 원본: 20261113/mentor/01_basic/common/support.py (그대로 포함) ──
"""강의 공통 오프라인 계산 도구: 실제 사전학습 LLM이 아닙니다.

교육용 정규표현식 토큰화·작은 벡터·가중치 계산만 제공합니다.
Transformer의 tokenizer/embedding 품질을 이 도구로 평가할 수 없습니다.
"""
from pathlib import Path
import json
import math
import re
import torch

DATA = [
    {"id":"A", "title":"실습 파일 보관", "content":"실습 파일은 개인 폴더에 보관합니다.", "category":"learning"},
    {"id":"B", "title":"서버 실행", "content":"프로젝트 폴더에서 서버를 실행합니다.", "category":"environment"},
    {"id":"C", "title":"질문 작성", "content":"오류 메시지와 시도한 작업을 함께 적습니다.", "category":"learning"},
]
SPECIAL = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

def words(text: str) -> list[str]:
    """교육용: 한글 연속 글자/영문/숫자/문장부호 분리. 서브워드 토크나이저가 아님."""
    return re.findall(r'[가-힣]+|[A-Za-z]+|\d+|[^\w\s]', text)

class ToyTokenizer:
    """BERT tokenizer 대체가 아닌, 오프라인 개념 확인용 규칙 기반 어휘표."""
    def __init__(self, corpus):
        wordset=sorted({word for line in corpus for word in words(line)})
        self.vocab={tok:i for i,tok in enumerate(SPECIAL + wordset)}
        self.reverse={i:t for t,i in self.vocab.items()}

    def encode(self, text: str, max_length=None, padding_to=None):
        raw=words(text)
        tokens=["[BOS]"]+raw+["[EOS]"]
        truncated=False
        if max_length is not None:
            if max_length<2: raise ValueError("BOS/EOS를 포함하려면 max_length>=2")
            if len(tokens)>max_length:
                tokens=tokens[:max_length-1]+["[EOS]"]
                truncated=True
        valid=len(tokens)
        if padding_to is not None:
            if padding_to<valid: raise ValueError("padding_to must be >= valid length")
            tokens=tokens+["[PAD]"]*(padding_to-valid)
        ids=[self.vocab.get(t,self.vocab["[UNK]"]) for t in tokens]
        mask=[0 if t=="[PAD]" else 1 for t in tokens]
        return {"tokens":tokens,"input_ids":ids,"attention_mask":mask,
                "valid_length":valid,"truncated":truncated,"original_length":len(raw)+2}

    def batch(self, texts, max_length=None):
        items=[self.encode(t,max_length=max_length) for t in texts]
        max_size=max(len(i["input_ids"]) for i in items)
        return [self.encode(t,max_length=max_length,padding_to=max_size) for t in texts]

def context_budget(window, parts:dict, output_reserved:int):
    used=sum(parts.values())
    return {"window":window,"input_used":used,"output_reserved":output_reserved,
            "remaining":window-used-output_reserved,
            "fits":used+output_reserved<=window}

def cosine(a,b):
    a=torch.as_tensor(a,dtype=torch.float32)
    b=torch.as_tensor(b,dtype=torch.float32)
    return float(torch.nn.functional.cosine_similarity(a,b,dim=0))

def attention(q,k,v,causal=True):
    q=torch.as_tensor(q,dtype=torch.float32)
    k=torch.as_tensor(k,dtype=torch.float32)
    v=torch.as_tensor(v,dtype=torch.float32)
    scores=q@k.T/math.sqrt(q.shape[-1])
    mask=torch.triu(torch.ones_like(scores,dtype=torch.bool),diagonal=1)
    weights=torch.softmax(scores.masked_fill(mask,float('-inf')) if causal else scores,dim=-1)
    return scores,weights,weights@v,mask

def write_json(path, data):
    p=Path(path);p.parent.mkdir(exist_ok=True,parents=True)
    p.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')

def write_md(path,text):
    p=Path(path);p.parent.mkdir(exist_ok=True,parents=True)
    p.write_text(text,encoding='utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

# ── 노트북 전용 준비 ──
import pandas as pd
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

import base64
from IPython.display import Image, display

def show_mermaid(mermaid_code):
    """Mermaid 텍스트를 그림으로 보여 줌 (mermaid.ink 이미지, 인터넷 필요)"""
    encoded = base64.urlsafe_b64encode(mermaid_code.encode("utf-8")).decode("ascii")
    display(Image(url="https://mermaid.ink/img/" + encoded))

print("준비 완료 ✅")

준비 완료 ✅


## B01. 구성요소 4개의 역할
역할을 섞지 않는 것이 핵심입니다. **DB는 다음 Token을 계산하지 않고, LLM은 영구 저장소를 대신하지 않습니다.**

In [2]:
roles = {
    "FastAPI": "HTTP 요청을 검증하고 함수·응답을 연결한다",
    "PostgreSQL": "원본 지식·분류를 관계형 테이블에 저장한다",
    "LLM": "문맥과 입력을 바탕으로 다음 Token을 생성한다",
    "Response": "호출한 Client에 JSON 결과를 전달한다",
}

pd.DataFrame(roles.items(), columns=["구성요소", "역할"])

,구성요소,역할
0,FastAPI,HTTP 요청을 검증하고 함수·응답을 연결한다
1,PostgreSQL,원본 지식·분류를 관계형 테이블에 저장한다
2,LLM,문맥과 입력을 바탕으로 다음 Token을 생성한다
3,Response,호출한 Client에 JSON 결과를 전달한다


In [3]:
assert set(roles) == {"FastAPI", "PostgreSQL", "LLM", "Response"}
assert all(text.strip() for text in roles.values())
print("B01 통과 ✅ 역할", len(roles), "개 작성")

B01 통과 ✅ 역할 4 개 작성


## B02. 요청 → 응답 연결 순서
`(출발, 도착, 설명)` 튜플 하나가 화살표 하나입니다. LLM이 들어간 연결은 **아직 만들지 않은 후속 단계**라서 설명에 `(후속 구현)`을 붙입니다.

In [4]:
edges = [
    ("Client", "FastAPI", "HTTP 요청"),
    ("FastAPI", "PostgreSQL", "원본 지식 조회·수정"),
    ("PostgreSQL", "FastAPI", "원본 데이터"),
    ("FastAPI", "LLM", "입력·문맥(후속 구현)"),
    ("LLM", "Response", "생성 결과(후속 구현)"),
    ("Response", "Client", "JSON"),
]
len(edges)

6

In [5]:
table = pd.DataFrame(edges, columns=["출발", "도착", "설명"])
table["상태"] = ["후속" if "후속" in label else "구현됨" for label in table["설명"]]
table

,출발,도착,설명,상태
0,Client,FastAPI,HTTP 요청,구현됨
1,FastAPI,PostgreSQL,원본 지식 조회·수정,구현됨
2,PostgreSQL,FastAPI,원본 데이터,구현됨
3,FastAPI,LLM,입력·문맥(후속 구현),후속
4,LLM,Response,생성 결과(후속 구현),후속
5,Response,Client,JSON,구현됨


요청은 Client에서 시작해 Client로 돌아와야 합니다.

In [6]:
assert edges[0][0] == "Client" and edges[-1][1] == "Client"
future = [f"{a}→{b}" for a, b, label in edges if "후속" in label]
print("후속 단계로 표시된 연결:", future)
assert all("LLM" in f for f in future)
print("B02 통과 ✅")

후속 단계로 표시된 연결: ['FastAPI→LLM', 'LLM→Response']
B02 통과 ✅


## B03. Mermaid 그림 만들기
Mermaid는 **글자로 쓰면 그림이 되는 표기법**입니다. 노드 이름에 `N0`, `N1` 같은 ID를 붙이고, 노드 줄과 화살표 줄을 차례로 만듭니다.

### B03-1. 노드 모으기와 ID 붙이기

In [7]:
edges = [
    ("Client", "FastAPI", "HTTP 요청"),
    ("FastAPI", "PostgreSQL", "원본 지식 조회·수정"),
    ("PostgreSQL", "FastAPI", "원본 데이터"),
    ("FastAPI", "LLM", "입력·문맥(후속 구현)"),
    ("LLM", "Response", "생성 결과(후속 구현)"),
    ("Response", "Client", "JSON"),
]
node_names = sorted({n for a, b, _ in edges for n in (a, b)})
node_names

['Client', 'FastAPI', 'LLM', 'PostgreSQL', 'Response']

In [8]:
node_id = {name: f"N{i}" for i, name in enumerate(node_names)}
node_id

{'Client': 'N0',
 'FastAPI': 'N1',
 'LLM': 'N2',
 'PostgreSQL': 'N3',
 'Response': 'N4'}

### B03-2. Mermaid 줄 만들기

In [9]:
lines = ["flowchart LR"]                                                   # 왼쪽 → 오른쪽 그림
lines += [f'  {node_id[name]}["{name}"]' for name in node_names]            # 노드 줄
lines += [f'  {node_id[a]} -->|"{label}"| {node_id[b]}' for a, b, label in edges]  # 화살표 줄
print("\n".join(lines))

flowchart LR
  N0["Client"]
  N1["FastAPI"]
  N2["LLM"]
  N3["PostgreSQL"]
  N4["Response"]
  N0 -->|"HTTP 요청"| N1
  N1 -->|"원본 지식 조회·수정"| N3
  N3 -->|"원본 데이터"| N1
  N1 -->|"입력·문맥(후속 구현)"| N2
  N2 -->|"생성 결과(후속 구현)"| N4
  N4 -->|"JSON"| N0


### B03-3. 그림으로 보기

In [10]:
show_mermaid("\n".join(lines))

### ✏️ B03 직접 해 보기 — 연결을 하나 추가해 보세요

In [11]:
#@title ✏️ 새 연결을 추가해 보세요 (예: 후속 수업의 Retriever)
source = "FastAPI"  #@param {type:"string"}
target = "Retriever"  #@param {type:"string"}
label = "검색 요청(후속 구현)"  #@param {type:"string"}

edges = [
    ("Client", "FastAPI", "HTTP 요청"),
    ("FastAPI", "PostgreSQL", "원본 지식 조회·수정"),
    ("PostgreSQL", "FastAPI", "원본 데이터"),
    ("FastAPI", "LLM", "입력·문맥(후속 구현)"),
    ("LLM", "Response", "생성 결과(후속 구현)"),
    ("Response", "Client", "JSON"),
]
my_edges = edges + [(source, target, label)]
names = sorted({n for a, b, _ in my_edges for n in (a, b)})
ids = {name: f"N{i}" for i, name in enumerate(names)}
my_lines = ["flowchart LR"]
my_lines += [f'  {ids[n]}["{n}"]' for n in names]
my_lines += [f'  {ids[a]} -->|"{t}"| {ids[b]}' for a, b, t in my_edges]
show_mermaid("\n".join(my_lines))

## 저장. 결과 저장 (`main.py`와 같은 결과)
`outputs/architecture.md`(Mermaid 그림)와 `outputs/architecture.json`(역할·연결 요약)을 저장합니다. 이 셀은 준비 셀만 있으면 혼자 실행됩니다.

In [12]:
roles = {
    "FastAPI": "HTTP 요청을 검증하고 함수·응답을 연결한다",
    "PostgreSQL": "원본 지식·분류를 관계형 테이블에 저장한다",
    "LLM": "문맥과 입력을 바탕으로 다음 Token을 생성한다",
    "Response": "호출한 Client에 JSON 결과를 전달한다",
}
edges = [
    ("Client", "FastAPI", "HTTP 요청"),
    ("FastAPI", "PostgreSQL", "원본 지식 조회·수정"),
    ("PostgreSQL", "FastAPI", "원본 데이터"),
    ("FastAPI", "LLM", "입력·문맥(후속 구현)"),
    ("LLM", "Response", "생성 결과(후속 구현)"),
    ("Response", "Client", "JSON"),
]
responsibility = {"storage": "DB는 원본 저장", "generation": "LLM은 언어 생성"}
trust_boundary = {"scope": "후속 구현 개념도"}

node_names = sorted({n for a, b, _ in edges for n in (a, b)})
node_id = {name: f"N{i}" for i, name in enumerate(node_names)}
lines = ["flowchart LR"]
lines += [f'  {node_id[name]}["{name}"]' for name in node_names]
lines += [f'  {node_id[a]} -->|"{label}"| {node_id[b]}' for a, b, label in edges]
write_md(OUT_DIR / "architecture.md", "# AI 지식 플랫폼 개념도\n\n```mermaid\n" + "\n".join(lines) + "\n```\n")
write_json(OUT_DIR / "architecture.json", {
    "roles": roles, "edges": [list(x) for x in edges],
    "responsibility": responsibility, "trust_boundary": trust_boundary,
})
print("[01] architecture.md / architecture.json 생성 완료")

[01] architecture.md / architecture.json 생성 완료


In [13]:
print((OUT_DIR / "architecture.md").read_text(encoding="utf-8"))

# AI 지식 플랫폼 개념도

```mermaid
flowchart LR
  N0["Client"]
  N1["FastAPI"]
  N2["LLM"]
  N3["PostgreSQL"]
  N4["Response"]
  N0 -->|"HTTP 요청"| N1
  N1 -->|"원본 지식 조회·수정"| N3
  N3 -->|"원본 데이터"| N1
  N1 -->|"입력·문맥(후속 구현)"| N2
  N2 -->|"생성 결과(후속 구현)"| N4
  N4 -->|"JSON"| N0
```



In [14]:
#@title 📥 결과 파일 내려받기 (Colab)
if IN_COLAB:
    from google.colab import files
    for name in ['architecture.md', 'architecture.json']:
        files.download(str(OUT_DIR / name))
else:
    print("Colab에서 실행하면 결과 파일이 내려받아집니다:", ['architecture.md', 'architecture.json'])

Colab에서 실행하면 결과 파일이 내려받아집니다: ['architecture.md', 'architecture.json']


---
### 정리
- **FastAPI**는 요청·응답 연결, **PostgreSQL**은 원본 저장, **LLM**은 다음 Token 생성, **Response**는 결과 전달.
- 오늘 실제로 동작하는 것은 Client ↔ FastAPI ↔ PostgreSQL까지이고, **LLM 연결은 후속 단계**다.
- Mermaid는 그림을 그리는 텍스트 표기법이지 API 호출이 아니다.